# 06 - Deployment
**CRISP-DM phase:** Deployment

The evaluated models are delivered as **Vehicle Safety Intelligence**, a web application with three human-review workflows. Each deployed model receives exactly the inputs it was trained and evaluated on.

| Workflow | Input | Models used | Answers |
|---|---|---|---|
| Complaint triage | a complaint narrative | U1 + U2 (Linear SVM) | Which component group(s)? Does it describe a crash, fire, injury or death? |
| Crash analysis | a police crash record in CRSS format (50 fields from 12 tables) | final S1 (15 features) + S2 (75 features) | How likely is injury, and serious or fatal injury, given what was recorded? Which fields drive the score? How does it compare with the recorded outcome? |
| Vehicle lookup | make, model, model year | linked vehicle profiles | How does this vehicle compare once complaints are set against crash involvement (a proxy for exposure)? |

| Component | Location | Hosting |
|---|---|---|
| Model API (FastAPI) | `app/api/main.py`, `app/api/crash_features.py` | Render: https://vehicle-safety-api.onrender.com |
| Web interface (HTML/JS) | `app/web/` | Vercel: https://vehicle-safety-intelligence-web.vercel.app |
| Artifact export and checks | `src/export_artifacts.py` | run before each release |
| Automated tests | `tests/test_api.py` | run locally before each release |

> **Initial plan:** a Streamlit dashboard calling the models directly (proposal and the first version of this notebook).
> **What we found:** a dashboard that holds the models itself can only be used through that dashboard, and free Streamlit hosting suits demonstrations rather than a service other tools could call.
> **Decision:** separate the **model API** (FastAPI, on Render) from the **interface** (static web page, on Vercel). Any system, such as a complaint-intake form or a case-management tool, can call the same API, and the interface stays lightweight.

## 6.0 Setup
The API is loaded in-process with FastAPI's test client, so every request below runs exactly the code that is deployed.

In [1]:
from pathlib import Path
import json, subprocess, sys

import joblib
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
API_DIR = PROJECT_ROOT / "app" / "api"
sys.path.insert(0, str(API_DIR))
from fastapi.testclient import TestClient
import main as api

client = TestClient(api.app)
ART = API_DIR / "artifacts"

## 6.1 Deployment artifacts
Only what the application needs is deployed: the two text models (each with its TF-IDF vocabulary), the two final crash models, the crash-record specification, a few real 2024 crashes for the demonstration, the vehicle profiles and a model card of test results. Raw data (6 GB) stays out of the deployment.

`src/export_artifacts.py` writes these files. Before it does, it checks that the API's feature builder (`app/api/crash_features.py`) reproduces the training features exactly for 3,000 real 2024 crashes; if any feature differs, the export stops.

In [2]:
what = {"text_U1.joblib": "U1 routing: TF-IDF vocabulary + 19 one-vs-rest Linear SVMs + thresholds",
        "text_U2.joblib": "U2 serious-incident flag: TF-IDF vocabulary + Linear SVM + threshold",
        "crash_y_injury.joblib": "final S1 pipeline (15 features from 13 fields) + threshold",
        "crash_y_serious.joblib": "final S2 pipeline (75 features from 47 fields) + threshold",
        "crash_spec.json": "the 50 required CRSS fields, official titles, unknown-value codes, training-year code lists",
        "crash_examples.json": "six real 2024 crashes (the test year) as CRSS records, for the demonstration",
        "vehicle_profiles.parquet": "make/model/year complaint and CRSS exposure profiles",
        "model_card.json": "headline test metrics shown on the Overview page"}
pd.DataFrame([{"file": f.name, "MB": round(f.stat().st_size / 1e6, 2), "contents": what.get(f.name, "")}
              for f in sorted(ART.iterdir())])

,file,MB,contents
0,crash_examples.json,0.14,six real 2024 crashes (the test year) as CRSS ...
1,crash_spec.json,0.01,"the 50 required CRSS fields, official titles, ..."
2,crash_y_injury.joblib,0.25,final S1 pipeline (15 features from 13 fields)...
3,crash_y_serious.joblib,0.64,final S2 pipeline (75 features from 47 fields)...
4,model_card.json,0.00,headline test metrics shown on the Overview page
5,text_U1.joblib,47.78,U1 routing: TF-IDF vocabulary + 19 one-vs-rest...
6,text_U2.joblib,6.17,U2 serious-incident flag: TF-IDF vocabulary + ...
7,vehicle_profiles.parquet,0.44,make/model/year complaint and CRSS exposure pr...


## 6.2 Complaint triage through the API

In [3]:
narrative = ("I was rear-ended at a stop light and the air bags did not deploy. My head hit the steering wheel "
             "and I was taken to the hospital by ambulance. A police report was filed.")
r = client.post("/api/complaint/analyze", json={"text": narrative}).json()
print("Serious-incident flag:", r["serious_incident"]["flagged"], "| score", r["serious_incident"]["score"],
      "| threshold", r["serious_incident"]["threshold"])
print("Evidence:", [e["term"] for e in r["serious_incident"]["evidence"]])
pd.DataFrame([{"component": c["component"], "score": c["score"], "threshold": c["threshold"],
               "recommended": c["recommended"], "evidence": ", ".join(e["term"] for e in c["evidence"][:5])}
              for c in r["components"]])

Serious-incident flag: True | score 3.048 | threshold -0.1775
Evidence: ['hit', 'was rear', 'hospital', 'bags did', 'rear ended', 'ended', 'police', 'police report']


,component,score,threshold,recommended,evidence
0,AIR BAGS,1.7888,0.0105,True,"bags, air bags, air, deploy, not deploy"
1,STEERING,-0.4906,-0.1952,False,"steering, the steering, wheel, police, hospital"
2,DRIVER ASSISTANCE,-1.0648,-0.6512,False,"rear ended, wheel and, light, filed, was rear"
3,EQUIPMENT,-1.0672,-0.5956,False,"the air, my head, light and, wheel and, air"
4,SEAT BELTS,-0.7771,-0.2548,False,"head hit, was rear, light and, my head, hit the"


**Interpretation.** The narrative is routed to **AIR BAGS** and flagged for priority review. Each decision comes with the terms that pushed it (term weight multiplied by its TF-IDF value in this narrative), so a reviewer can see why.

### Round-trip check
The score returned by the API must equal the score from loading the saved model directly; this confirms the deployed service uses the validated model unchanged.

In [4]:
u2 = joblib.load(ART / "text_U2.joblib")
direct = float(u2["model"].decision_function(u2["vectorizer"].transform([narrative]))[0])
print(f"direct {direct:.6f} | via API {r['serious_incident']['score']:.6f} | equal: {abs(direct - r['serious_incident']['score']) < 1e-4}")

direct 3.047994 | via API 3.048000 | equal: True


## 6.3 Crash analysis through the API
**Input format.** One crash per request: its rows from the 12 required CRSS tables, in CRSS codes and with the CRSS field names, nested by table. The API reads only the 50 fields the final models use and ignores everything else, so rows taken straight from a full CRSS release work as they are. A table given as an empty list means the crash has no such rows (for example no pedestrians); a table left out is treated as missing and flagged.

In [5]:
schema = client.get("/api/crash/schema").json()
print(schema["format"])
print(f"\n{schema['n_fields']} fields from {len(schema['tables'])} tables:")
for table, fields in schema["tables"].items():
    print(f"  {table:9s} {', '.join(f['field'] for f in fields)}")
print("\nKeys:", schema["keys"])

One crash per request: {'record': {table: [rows]}}, CRSS codes, field names as in the CRSS files. A table given as an empty list means the crash has no such rows; an absent table is treated as missing.

50 fields from 12 tables:
  accident  ALCHL_IM, EVENT1_IM, HOUR_IM, LGTCON_IM, MANCOL_IM, REGION, RELJCT2_IM, URBANICITY
  cevent    AOI1, AOI2, SOE, n_rows
  distract  DRDISTRACT
  drimpair  DRIMPAIR
  driverrf  DRIVERRF
  maneuver  MANEUVER
  pbtype    PEDPOS
  person    AGE_IM, AIR_BAG, BODY_TYP, DRUGS, EJECT_IM, GVWR_FROM, IMPACT1, MOD_YEAR, PERALCH_IM, PER_TYP, REST_MIS, REST_USE, SEX_IM, n_rows
  personrf  PERSONRF
  vehicle   BODY_TYP, HIT_RUN, IMPACT1_IM, MDLYR_IM, PCRASH1_IM, P_CRASH2, SPEEDREL, TRAV_SP, VEVENT_IM, VNUM_LAN, VPROFILE, VSPD_LIM, VSURCOND, VTRAFCON, VTRAFWAY, V_CONFIG
  vevent    n_rows
  violatn   VIOLATION

Keys: {'CASENUM': 'crash number; groups rows into one crash when a file holds many crashes', 'VEH_NO': 'vehicle number; labels vehicle-level rows (optional 

**Demonstration crashes.** Six real crashes from 2024, the test year the models never trained on, chosen to show the range of results, each picked by a fixed rule in `src/export_artifacts.py` (the middle-scoring crash in its category) rather than by hand: a low-risk crash, an injury crash, two serious crashes the models flag, one serious crash they miss, and one false alarm. Each is sent to the API exactly as a CRSS record; the recorded outcome is compared afterwards.

In [6]:
rows = []
for ex in client.get("/api/crash/examples").json():
    e = client.get(f"/api/crash/examples/{ex['id']}").json()
    r = client.post("/api/crash/assess", json={"record": e["record"]}).json()
    rows.append({"crash": e["casenum"], "chosen as": e["label"],
                 "injury prob.": r["injury_crash"]["probability"], "injury flag": r["injury_crash"]["flagged"],
                 "serious prob.": r["serious_or_fatal_crash"]["probability"], "serious flag": r["serious_or_fatal_crash"]["flagged"],
                 "recorded outcome": e["recorded_outcome"], "missing fields": len(r["missing_fields"])})
pd.DataFrame(rows)

,crash,chosen as,injury prob.,injury flag,serious prob.,serious flag,recorded outcome,missing fields
0,202406024830,No injury recorded; low predicted risk,0.0836,False,0.0030,False,No Apparent Injury (O),0
1,202405818149,"Injury, not serious; injury flagged",0.8513,True,0.0453,False,Possible Injury (C),0
2,202405637028,Serious or fatal; high predicted risk,0.9064,True,0.9005,True,Suspected Serious Injury (A),0
3,202406117886,Serious or fatal; serious flagged,0.9575,True,0.3998,True,Suspected Serious Injury (A),0
4,202405811016,"Serious or fatal, but scored below the thresho...",0.9892,True,0.0832,False,Suspected Serious Injury (A),0
5,202406093754,"No injury recorded, but flagged (a false alarm)",0.8624,True,0.2002,False,No Apparent Injury (O),0


In [7]:
e = client.get("/api/crash/examples/crash-4").json()
print(f"Crash {e['casenum']} as sent: " + ", ".join(f"{t} ({len(rows)} rows)" for t, rows in e["record"].items()))
print("accident row:", e["record"]["accident"][0])
r = client.post("/api/crash/assess", json={"record": e["record"]}).json()
for key in ("injury_crash", "serious_or_fatal_crash"):
    x = r[key]
    print(f"\n{key}: {x['probability']:.1%} (threshold {x['threshold']:.1%}, flagged {x['flagged']}) "
          f"- {x['n_features']} features from {x['n_fields']} fields")
    for f in x["factors"]:
        print(f"   {f['effect']:+.3f}  {f['label']}")
print("\nRecorded outcome:", e["recorded_outcome"])

Crash 202406117886 as sent: accident (1 rows), cevent (5 rows), distract (2 rows), drimpair (2 rows), driverrf (3 rows), maneuver (2 rows), pbtype (0 rows), person (3 rows), personrf (3 rows), vehicle (2 rows), vevent (6 rows), violatn (2 rows)
accident row: {'CASENUM': 202406117886, 'ALCHL_IM': 1, 'EVENT1_IM': 12, 'HOUR_IM': 21, 'LGTCON_IM': 3, 'MANCOL_IM': 8, 'REGION': 4, 'RELJCT2_IM': 1, 'URBANICITY': 1}

injury_crash: 95.8% (threshold 40.5%, flagged True) - 15 features from 13 fields
   +0.077  Air Bag Deployed
   +0.052  Sequence of Events
   +0.037  Number of vehicle events
   -0.026  Alcohol Involved in Crash
   +0.013  Police Reported Alcohol Involvement
   +0.010  Total Lanes in Roadway

serious_or_fatal_crash: 40.0% (threshold 25.2%, flagged True) - 75 features from 47 fields
   +0.233  Area of Impact (Other Vehicle)
   +0.200  Air Bag Deployed
   -0.192  Total Lanes in Roadway
   +0.129  Police Reported Alcohol Involvement
   +0.107  Speeding Related
   -0.094  Police Report

**Interpretation.** The serious crashes score far higher than the low-risk one on both targets, and the table includes a miss and a false alarm, so the demonstration shows the models' limits as well as their strengths. The **factors** are a local explanation: for each field, the change in probability if that field were unknown (the model then uses the training median, or no category, for that field's features). Every response says that probabilities are relative to the CRSS sample, which over-represents injury crashes.

### Trained = evaluated = deployed
The score the API returns from a raw CRSS record must equal the score the saved model gives on that crash's row in the training feature table (`data/processed/crss_crash_features.parquet`). This confirms that the deployed feature builder and the evaluated model are the ones from notebooks 04 and 05.

In [8]:
stored = json.loads((ART / "crash_examples.json").read_text())["examples"]
checks = []
for e in stored:
    r = client.post("/api/crash/assess", json={"record": e["record"]}).json()
    for target, key in [("y_injury", "injury_crash"), ("y_serious", "serious_or_fatal_crash")]:
        b = joblib.load(ART / f"crash_{target}.joblib")
        direct = float(b["pipeline"].predict_proba(pd.DataFrame([e["training_features"]])[b["features"]].astype(float))[:, 1][0])
        checks.append({"crash": e["casenum"], "model": target, "API": r[key]["probability"], "training features": round(direct, 4)})
checks = pd.DataFrame(checks)
print("All scores match:", bool((checks["API"] - checks["training features"]).abs().max() < 1e-4))
checks

All scores match: True


,crash,model,API,training features
0,202406024830,y_injury,0.0836,0.0836
1,202406024830,y_serious,0.0030,0.0030
2,202405818149,y_injury,0.8513,0.8513
3,202405818149,y_serious,0.0453,0.0453
4,202405637028,y_injury,0.9064,0.9064
5,202405637028,y_serious,0.9005,0.9005
6,202406117886,y_injury,0.9575,0.9575
7,202406117886,y_serious,0.3998,0.3998
8,202405811016,y_injury,0.9892,0.9892
9,202405811016,y_serious,0.0832,0.0832


## 6.4 Vehicle lookup through the API

In [9]:
p = client.get("/api/vehicles/profile", params={"make": "FORD", "model": "ESCAPE", "year": 2017}).json()
print(json.dumps({k: p[k] for k in ("vehicle", "integrated")}, indent=1))
print("Top components:", dict(list(p["complaints"]["by_component"].items())[:4]))
pd.DataFrame(client.get("/api/vehicles/top", params={"limit": 10}).json())

{
 "vehicle": {
  "make": "FORD",
  "model": "ESCAPE",
  "model_year": 2017
 },
 "integrated": {
  "complaints_per_10k_crash_involved": 287.1357,
  "percentile_among_vehicles": 0.9671,
  "reliable": true
 }
}
Top components: {'ENGINE': 1125, 'POWER TRAIN': 262, 'ELECTRICAL': 68, 'WHEELS': 61}


,make,model,model_year,complaints,complaints_per_10k_crash_involved,serious_share
0,CHEVROLET,VOLT,2017,519,1968.1,0.0231
1,FORD,MUSTANG MACH E,2021,263,1470.4,0.0380
2,TESLA,MODEL S,2016,381,1319.2,0.0787
3,TESLA,MODEL S,2017,306,990.2,0.0523
4,HONDA,HR-V,2023,632,960.1,0.0047
5,ACURA,INTEGRA,2023,196,944.7,0.0153
6,RAM,2500,2018,1861,842.8,0.0118
7,SUBARU,OUTBACK,2020,828,779.5,0.0229
8,CHEVROLET,VOLT,2018,156,768.1,0.0128
9,TESLA,MODEL 3,2018,762,743.4,0.1050


**Interpretation.** This is the integration of the two datasets: complaints from the owner-complaint file divided by CRSS-estimated crash involvement for the same make, model and year. The 2017 Ford Escape is in the 97th percentile, with engine complaints dominating, which is consistent with its publicly reported engine-coolant problems.

## 6.5 Input validation and safeguards

In [10]:
ex = client.get("/api/crash/examples/crash-1").json()["record"]
two_crashes = json.loads(json.dumps(ex)); two_crashes["vehicle"][0]["CASENUM"] = 1
no_person = {t: rows for t, rows in ex.items() if t != "person"}
with_outcome = json.loads(json.dumps(ex)); with_outcome["accident"][0]["MAX_SEV"] = 4
checks = [
    ("narrative shorter than 20 characters", client.post("/api/complaint/analyze", json={"text": "too short"}).status_code),
    ("crash record mixing rows from two crashes", client.post("/api/crash/assess", json={"record": two_crashes}).status_code),
    ("crash record with none of the required tables", client.post("/api/crash/assess", json={"record": {"weather": []}}).status_code),
    ("vehicle with no complaints", client.get("/api/vehicles/profile", params={"make": "FORD", "model": "NO SUCH MODEL", "year": 2017}).status_code),
]
display(pd.DataFrame(checks, columns=["request", "HTTP status"]))
r = client.post("/api/crash/assess", json={"record": no_person}).json()
print("Crash without the person table ->", r["warning"])
print("   missing:", [m["field"] for m in r["missing_fields"]])
r = client.post("/api/crash/assess", json={"record": with_outcome}).json()
print("Crash record containing MAX_SEV -> ignored:", r["ignored_outcome_fields"])
print("Disclaimer on every response ->", r["disclaimer"])

,request,HTTP status
0,narrative shorter than 20 characters,422
1,crash record mixing rows from two crashes,422
2,crash record with none of the required tables,422
3,vehicle with no complaints,404


Crash without the person table -> 14 of 50 required fields were not supplied and were treated as unknown; confidence is reduced.
   missing: ['person.AGE_IM', 'person.AIR_BAG', 'person.BODY_TYP', 'person.DRUGS', 'person.EJECT_IM', 'person.GVWR_FROM', 'person.IMPACT1', 'person.MOD_YEAR', 'person.PERALCH_IM', 'person.PER_TYP', 'person.REST_MIS', 'person.REST_USE', 'person.SEX_IM', 'person.n_rows']
Crash record containing MAX_SEV -> ignored: ['MAX_SEV']
Disclaimer on every response -> Decision support for human review only. Outputs do not establish causation, liability or the existence of a defect.


**Interpretation.** Invalid requests are rejected with clear errors (422 = invalid input, 404 = not found) instead of producing a misleading prediction. A record with missing tables is still scored, but never silently: the missing fields are listed and a warning says confidence is reduced. Inside the model they are filled exactly as in training (the training median for numbers and counts, an unknown category for crash-level codes). Outcome fields such as MAX_SEV are never used as inputs: if a record contains them they are ignored and named in the response. Every response carries the human-review disclaimer, and browser access to the deployed API is restricted to the project's own website.

## 6.6 Automated test suite
The same checks, plus others, run as automated tests before each release.

In [11]:
res = subprocess.run([sys.executable, "-m", "pytest", "tests", "-q", "-p", "no:cacheprovider", "--color=no"],
                     cwd=PROJECT_ROOT, capture_output=True, text=True)
print(res.stdout[-600:])

                               [100%]
============================== warnings summary ===============================
.venv\Lib\site-packages\starlette\testclient.py:51
  C:\Users\delan\Downloads\NHTSA_Road_Vehicle_Safety_Capstone_Last_Lap\.venv\Lib\site-packages\starlette\testclient.py:51: DeprecationWarning: The anyio.abc.BlockingPortal alias is deprecated, use anyio.from_thread.BlockingPortal instead.
    _PortalFactoryType = Callable[[], AbstractContextManager[anyio.abc.BlockingPortal]]

-- Docs: https://docs.pytest.org/en/stable/how-to/capture-warnings.html
13 passed, 1 warning in 14.49s



## 6.7 Live deployment check
Free hosting puts the API to sleep after 15 minutes without traffic, so the first request can take up to a minute.

The hosted service is updated by redeploying `app/api` (Render) and `app/web` (Vercel) after `src/export_artifacts.py` and the tests pass. If the model card below still shows the benchmark figures (S1 ROC-AUC 0.906), the hosted copy predates the final crash models and the CRSS-record crash tool, and needs redeploying.

In [12]:
import urllib.request
for url in ["https://vehicle-safety-api.onrender.com/health", "https://vehicle-safety-api.onrender.com/api/meta"]:
    try:
        with urllib.request.urlopen(url, timeout=90) as resp:
            print(url, "->", resp.status, resp.read(160).decode())
    except Exception as e:
        print(url, "-> not reachable from this machine right now:", e)

https://vehicle-safety-api.onrender.com/health -> 200 {"status":"ok"}
https://vehicle-safety-api.onrender.com/api/meta -> 200 {"models":[{"task":"S1 Injury crash (CRSS)","model":"Gradient Boosting","headline_metric":"PR-AUC, 2024 test","headline_value":"0.894","detail":"baseline 0.513;


## 6.8 Operating, monitoring and retraining
| Concern | Plan |
|---|---|
| **Human oversight** | Outputs are recommendations; every response says they do not establish causation, liability or a defect. |
| **Drift** | Complaint mix changes over time (notebook 02). Track monthly: share of complaints flagged, distribution of recommended components, and agreement with analysts' final routing. Retrain text models yearly, or sooner if agreement falls. |
| **New CRSS years** | Each annual CRSS release: rebuild features with the training-year code lists, re-evaluate on the newest year before replacing models. |
| **Threshold policy** | Thresholds were set on validation data to balance precision and recall; an agency can lower them to catch more cases at the cost of more reviews. |
| **Reproducibility** | Every artifact is rebuilt by the scripts in `src/` (README); raw files are hashed (notebook 02). |
| **Privacy** | The application does not store submitted narratives or crash records; they are processed in memory only. The hosting platform keeps standard request logs (time, address, status), not request contents. |